# 単回帰分析

単回帰分析では、式を暗記するより、計算の意味をコードで検証する姿勢を優先します。


## 背景と目的

まず単回帰を理解すると、予測誤差とパラメータ推定の関係を最小構成で把握できます。

勾配や最小二乗の意味が明確になり、複雑なモデルへ進んだときの基準点になります。

直線モデルの仮定が当たる条件と外れる条件をコードで確かめます。


## 最初に解きたい疑問

1. このノートで `x` と `y` は何を表していて、どちらが入力でどちらが予測したい値なのか。
2. `w0` と `w1` はどうやって決まっていて、数字を見たときにどう解釈すればよいのか。
3. `residual` と `mse` は何が違っていて、どちらを見れば予測の良し悪しが分かるのか。
4. `x^2` を特徴量に足すとき、なぜ直線の説明から急に外れたように見えないのか。
5. `train_x, test_x` に分ける理由は何で、この小さいデータでも評価分割が必要なのか。


## 先に押さえる言葉

- `切片`: 直線が `y` 軸と交わる位置を表す値です。入力が0のときの基準点だと考えると分かりやすいです。
- `傾き`: 入力 `x` が1増えたときに出力 `y` がどれだけ増減するかを表します。値が大きいほど、直線は急になります。
- `残差`: 実際の値と予測値の差です。予測がどれだけ外れたかを1件ずつ見るための基本の量です。
- `平均二乗誤差`: 残差を二乗して平均したものです。大きい外れを強く罰するので、モデル全体のズレを1つの数字で表せます。


## 実行前の見取り図

1. `w0, w1` の出力が、元データの増え方とだいたい合っているかを見る。
2. `mse` と `test mae` が極端に大きくなっていないかを確認する。
3. `pred` と `residual` を見て、誤差が一方向に偏っていないかを確認する。


## 出力の読み方

- `test mae` の値を高い/低いのどちらで読めばよいか。


## つまずきやすい点

- `x_bar`, `y_bar`, `num`, `den` がそれぞれ何を意味するのかを、式だけでなく言葉でも分解する説明が必要です。
- 特徴量を増やす話と評価分割の話が、なぜ『学習した気分』と『本当に当たる』の差につながるのかを補う説明が必要です。
- toy split なので test mae が安定しない注意。
- 二乗特徴が線形モデルに何を追加するのかの説明。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- この分割は性能比較の結論ではなく、流れ確認用の例であること。


## 実験 1: 単回帰データを作る

単回帰の前提を確認するため、直線関係が見える最小データを作ります。


In [ ]:
x = [1, 2, 3, 4, 5]
y = [2.1, 4.2, 6.1, 8.2, 10.1]
pairs = list(zip(x, y))
print('task = simple-regression')
print('pairs =', pairs)

このデータを基準に、係数推定と誤差評価の流れを追います。



## 実験 2: 単回帰を手で計算する

次に、最小二乗法の係数を手計算で求めます。既存ライブラリを使う前に中身を一度体験すると理解が安定します。


In [ ]:
x_bar = sum(x) / len(x)
y_bar = sum(y) / len(y)
num = sum((xi - x_bar) * (yi - y_bar) for xi, yi in pairs)
den = sum((xi - x_bar) ** 2 for xi in x)
w1 = num / den; w0 = y_bar - w1 * x_bar
print('w0, w1 =', round(w0, 4), round(w1, 4))

式の記号は抽象的に見えますが、コードでは平均・差分・和に分解されます。難しい式ほど、実装で部品に分けて確認するのが有効です。



## 式と実装の往復

1. $\hat{y} = f_{\theta}(x)$
2. $L(\theta) = \frac{1}{N}\sum_i \ell(\hat{y}_i, y_i)$


## 実験 3: 予測と誤差を見る

ここで、求めた係数で予測を出し、誤差を数値化します。モデル改善は誤差の観察から始まります。


In [ ]:
pred = [w0 + w1 * xi for xi in x]
residual = [yi - pi for yi, pi in zip(y, pred)]
mse = sum((r ** 2) for r in residual) / len(residual)
print('pred =', pred)
print('mse =', round(mse, 8))

誤差を観察するときは、平均値だけでなく個別の残差も見てください。偏りがあると、モデル構造の見直しが必要になります。



## 実験 4: 特徴量を拡張する

次に、特徴量を追加したときに表現力がどう変わるかを確認します。特徴量設計は精度に直接効く実装作業です。


In [ ]:
x2 = [xi ** 2 for xi in x]
feature = list(zip(x, x2))
print('feature sample =', feature[:3])
scaled_x = [(xi - x_bar) / (max(x) - min(x)) for xi in x]
print('scaled_x =', [round(v, 4) for v in scaled_x])

特徴量を増やすと表現力は上がりますが、同時に過学習リスクも増えます。増やす理由を説明できる特徴だけを採用する姿勢が重要です。



## 実験 5: 評価分割を体験する

最後に、学習と評価を分ける意味をコードで確かめます。分割しない評価は自己採点に近く、実運用の性能を過大評価しがちです。


In [ ]:
train_x, test_x = x[:3], x[3:]
train_y, test_y = y[:3], y[3:]
pred_test = [w0 + w1 * xi for xi in test_x]
mae_test = sum(abs(yi - pi) for yi, pi in zip(test_y, pred_test)) / len(test_y)
print('test mae =', round(mae_test, 6))

検証データでの誤差が急に悪化したら、モデルより先にデータ分布の差を疑ってください。この視点は実務で非常に重要です。



## 振り返り

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 訓練データと評価データの分布差を見ない
2. 単一スコアだけでモデルを選んでしまう
3. 前処理の漏れを評価後に気づく
